# MetroStack Live Audit Notebook ⚙️🔬

**Portable dual-LLM functional audit for any deployed MetroStack instance.**

This notebook targets a live MetroStack API and validates all 7 feature tabs using a two-agent architecture:

- **Junior Agent** (Port 8080) — Generates test payloads and edge cases for each feature
- **Senior Agent** (Port 8081) — Receives raw API responses and grades correctness, schema compliance, and behaviour

No filesystem access required. No CrewAI dependency. Runs in standard Jupyter or MetroNote (Pyodide browser runtime).

---
### Feature Audit Scope
| ID | Feature | Key Behaviours |
|---|---|---|
| PROJ-01 | Project Management | CRUD, status lifecycle |
| FILE-01 | File / Scan Ingest | Upload, parse, point count |
| ALIGN-01 | Alignment | Best-fit, pin bore control, Z-offset |
| DEV-01 | Deviation Analysis | Per-point stats, min/max/mean/stddev |
| GDT-01 | GD&T Features | Tolerance evaluation, datum chain |
| WALL-01 | Wall Thickness | Section analysis, pass/fail bands |
| VIZ-01 | Visualization | Colour map, deviation overlay, export |

---
**Usage:** Edit `METROSTACK_URL`, `JUNIOR_URL`, `SENIOR_URL` in Cell 1 to point to your deployment, then Run All.

In [1]:
# ============================================================
# Cell 1 — Configuration & Pyodide Network Bridge
# ============================================================

import sys, json, re, time
from dataclasses import dataclass, field
from typing import Optional

# --- TARGET ENDPOINTS ---
METROSTACK_URL = "http://localhost:8000"   # FastAPI backend base URL
JUNIOR_URL     = "http://localhost:8080"   # Qwen Coder / smaller model
SENIOR_URL     = "http://localhost:8081"   # DeepSeek Coder / larger model

# --- SCORING THRESHOLDS ---
PASS_THRESHOLD  = 0.65   # senior grade >= 65% = PASS
RETRY_THRESHOLD = 0.40   # below 40% = FAIL (no retry benefit)
MAX_RETRIES     = 2

# --- PYODIDE DETECTION ---
# When running inside MetroNote (browser/Pyodide), network calls must
# route through the JavaScript fetch layer. In standard Jupyter they
# use the requests library directly.
PYODIDE_ENV = 'pyodide' in sys.modules

if PYODIDE_ENV:
    try:
        import micropip
        print("Installing browser runtime drivers...")
        await micropip.install(['pyodide-http'])
        import pyodide_http
        pyodide_http.patch_all()
        print("\u2713 Browser network engine bridged.")
    except Exception as e:
        print(f"Browser patch warning: {e}")
else:
    try:
        import requests
        print("\u2713 Standard Jupyter mode — requests library ready.")
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'requests'], check=True)
        import requests
        print("\u2713 requests installed and ready.")

print(f"""
============================================================
  MetroStack Audit Notebook — Configuration
  MetroStack API : {METROSTACK_URL}
  Junior Agent   : {JUNIOR_URL}  (port 8080)
  Senior Agent   : {SENIOR_URL}  (port 8081)
  Runtime        : {'Pyodide/Browser' if PYODIDE_ENV else 'Jupyter/Python'}
============================================================
""")

Installing browser runtime drivers...
✓ Browser network engine bridged.

  MetroStack Audit Notebook — Configuration
  MetroStack API : http://localhost:8000
  Junior Agent   : http://localhost:8080  (port 8080)
  Senior Agent   : http://localhost:8081  (port 8081)
  Runtime        : Pyodide/Browser



In [2]:
# ============================================================
# Cell 2 — MetroNote Dataset Scanner
#
# MetroNote uploads environment files into the Pyodide virtual
# filesystem. In standard Jupyter, files sit on disk next to
# the notebook. This cell searches every likely path for:
#
#   .json  — fixture datasets (projects, scans, alignments)
#   .csv   — tabular point-cloud or deviation exports
#   .e57   — scan file (presence only — binary, not parsed)
#   .xyz / .ply / .pts — point cloud text formats
#   .txt   — generic exports or ID lists
#
# Whatever is found seeds the audit with real IDs and payloads
# instead of synthetic placeholders.
# ============================================================

import os, json, csv, io, pathlib, sys

# ------------------------------------------------------------------
# Search paths — ordered by likelihood for each runtime.
# Pyodide (MetroNote browser) typically mounts uploaded files at /
# or /home/pyodide. Jupyter puts them next to the .ipynb file.
# ------------------------------------------------------------------
PYODIDE_ENV = 'pyodide' in sys.modules

if PYODIDE_ENV:
    SEARCH_ROOTS = [
        "/",              # Pyodide FS root — MetroNote drops files here
        "/home/pyodide",  # alternate Pyodide home
        "/uploads",       # some MetroNote builds use this explicitly
        "/tmp",
    ]
else:
    try:
        nb_dir = pathlib.Path(__file__).parent
    except NameError:
        nb_dir = pathlib.Path.cwd()
    SEARCH_ROOTS = [
        str(nb_dir),
        str(pathlib.Path.cwd()),
        str(nb_dir / 'datasets'),
        str(nb_dir / 'uploads'),
        str(nb_dir / 'fixtures'),
    ]

SCAN_EXTENSIONS = {'.json', '.csv', '.e57', '.xyz', '.ply', '.pts', '.txt'}
SKIP_SUFFIXES   = {'.gguf', '.bin', '.safetensors', '.onnx', '.ipynb', '.html', '.py', '.md'}
SKIP_NAMES      = {'.DS_Store', 'Thumbs.db'}


def scan_for_datasets(roots):
    found = []
    seen  = set()
    for root in roots:
        try:
            for entry in os.scandir(root):
                p = pathlib.Path(entry.path)
                if str(p) in seen:
                    continue
                seen.add(str(p))
                if p.name in SKIP_NAMES or p.suffix.lower() in SKIP_SUFFIXES:
                    continue
                if entry.is_file() and p.suffix.lower() in SCAN_EXTENSIONS:
                    found.append(p)
        except (PermissionError, FileNotFoundError, OSError):
            pass
    return found


def try_parse_json(path):
    try:
        with open(path, 'r', encoding='utf-8', errors='replace') as fh:
            return json.load(fh)
    except Exception:
        return None


def try_parse_csv(path):
    try:
        with open(path, 'r', encoding='utf-8', errors='replace') as fh:
            rows = list(csv.DictReader(fh))
        return rows if rows else None
    except Exception:
        return None


def try_parse_xyz(path, max_lines=5):
    try:
        lines, total = [], 0
        with open(path, 'r', encoding='utf-8', errors='replace') as fh:
            for line in fh:
                total += 1
                if total <= max_lines:
                    lines.append(line.strip())
        return {'sample_lines': lines, 'estimated_point_count': total}
    except Exception:
        return None


# ── ID / context store shared with all downstream cells ───────────────────────
DATASET_CONTEXT = {
    'project_ids':       [],
    'scan_ids':          [],
    'alignment_ids':     [],
    'deviation_ids':     [],
    'part_numbers':      [],
    'point_cloud_files': [],
    'raw_fixtures':      {},
    'summary':           [],
}


def _extract_ids(data, filename):
    """Walk parsed data (dict or list) and pull known ID/field names."""
    if isinstance(data, dict):
        for key in ('data', 'results', 'items', 'records'):
            if key in data and isinstance(data[key], list):
                data = data[key]
                break
    rows = data if isinstance(data, list) else [data]
    hint = filename.lower()

    bucket_fields = {
        'project_id':   'project_ids',
        'scan_id':      'scan_ids',
        'alignment_id': 'alignment_ids',
        'deviation_id': 'deviation_ids',
    }

    for row in rows:
        if not isinstance(row, dict):
            continue
        for field, bucket in bucket_fields.items():
            val = row.get(field)
            if val and str(val) not in DATASET_CONTEXT[bucket]:
                DATASET_CONTEXT[bucket].append(str(val))

        generic = row.get('id')
        if generic:
            gid = str(generic)
            if   'project' in hint and gid not in DATASET_CONTEXT['project_ids']:
                DATASET_CONTEXT['project_ids'].append(gid)
            elif 'scan'    in hint and gid not in DATASET_CONTEXT['scan_ids']:
                DATASET_CONTEXT['scan_ids'].append(gid)
            elif 'align'   in hint and gid not in DATASET_CONTEXT['alignment_ids']:
                DATASET_CONTEXT['alignment_ids'].append(gid)
            elif 'dev'     in hint and gid not in DATASET_CONTEXT['deviation_ids']:
                DATASET_CONTEXT['deviation_ids'].append(gid)

        pn = row.get('part_number')
        if pn and str(pn) not in DATASET_CONTEXT['part_numbers']:
            DATASET_CONTEXT['part_numbers'].append(str(pn))


# ── Run scan ───────────────────────────────────────────────────────────────────
print("Scanning for dataset files...")
print(f"  Runtime  : {'Pyodide/Browser' if PYODIDE_ENV else 'Jupyter/Python'}")
print(f"  Paths    : {SEARCH_ROOTS}")
print()

found_files = scan_for_datasets(SEARCH_ROOTS)

if not found_files:
    print("  No dataset files found.")
    print("  Upload .json fixture exports or .csv/.xyz point-cloud files")
    print("  via the MetroNote ENVIRONMENT FILES panel, then re-run this cell.")
    print()
    print("  ⚠  Audit will proceed using synthetic placeholder IDs.")
    print("     Tests still verify API structure and error-handling behaviour,")
    print("     but cannot validate against real MetroStack records.")
else:
    print(f"  Found {len(found_files)} file(s):")
    print()
    for p in found_files:
        ext      = p.suffix.lower()
        size_kb  = p.stat().st_size / 1024
        status   = ""

        if ext == '.json':
            data = try_parse_json(p)
            if data is not None:
                _extract_ids(data, p.name)
                DATASET_CONTEXT['raw_fixtures'][p.name] = data
                status = "✓ parsed"
            else:
                status = "⚠ parse failed"

        elif ext == '.csv':
            rows = try_parse_csv(p)
            if rows is not None:
                _extract_ids(rows, p.name)
                DATASET_CONTEXT['raw_fixtures'][p.name] = rows
                status = f"✓ {len(rows)} rows"
            else:
                status = "⚠ parse failed"

        elif ext in {'.e57', '.ply', '.pts'}:
            DATASET_CONTEXT['point_cloud_files'].append(str(p))
            status = "✓ registered (binary)"

        elif ext == '.xyz':
            parsed = try_parse_xyz(p)
            if parsed:
                DATASET_CONTEXT['point_cloud_files'].append(str(p))
                DATASET_CONTEXT['raw_fixtures'][p.name] = parsed
                status = f"✓ ~{parsed['estimated_point_count']} pts"
            else:
                status = "⚠ could not read"

        elif ext == '.txt':
            data = try_parse_json(p)
            if data:
                _extract_ids(data, p.name)
                DATASET_CONTEXT['raw_fixtures'][p.name] = data
                status = "✓ JSON-in-txt"
            else:
                rows = try_parse_csv(p)
                if rows:
                    _extract_ids(rows, p.name)
                    status = f"✓ CSV-in-txt ({len(rows)} rows)"
                else:
                    status = "~ plain text"

        line = f"    [{ext:5s}]  {p.name:<42s} {size_kb:>7.1f} KB  {status}"
        print(line)
        DATASET_CONTEXT['summary'].append(line.strip())

print()
print("  Extracted context:")
print(f"    project_ids       : {len(DATASET_CONTEXT['project_ids'])}  {DATASET_CONTEXT['project_ids'][:2]}")
print(f"    scan_ids          : {len(DATASET_CONTEXT['scan_ids'])}  {DATASET_CONTEXT['scan_ids'][:2]}")
print(f"    alignment_ids     : {len(DATASET_CONTEXT['alignment_ids'])}")
print(f"    deviation_ids     : {len(DATASET_CONTEXT['deviation_ids'])}")
print(f"    part_numbers      : {DATASET_CONTEXT['part_numbers'][:3]}")
print(f"    point_cloud_files : {len(DATASET_CONTEXT['point_cloud_files'])}")


# ── Helper functions used by all downstream audit cells ───────────────────────
def get_test_id(bucket, fallback='nonexistent-id-000'):
    """Return first real ID from the named bucket, or the fallback string."""
    ids = DATASET_CONTEXT.get(bucket, [])
    return ids[0] if ids else fallback

def get_part_number(fallback='AUDIT-PN-001'):
    pns = DATASET_CONTEXT.get('part_numbers', [])
    return pns[0] if pns else fallback

print()
print("✓ DATASET_CONTEXT and helpers ready for use by all downstream cells.")


Scanning for dataset files...
  Runtime  : Pyodide/Browser
  Paths    : ['/', '/home/pyodide', '/uploads', '/tmp']

  Found 1 file(s):

    [.json]  dataset_summary.json                           0.6 KB  ✓ parsed

  Extracted context:
    project_ids       : 0  []
    scan_ids          : 0  []
    alignment_ids     : 0
    deviation_ids     : 0
    part_numbers      : []
    point_cloud_files : 0

✓ DATASET_CONTEXT and helpers ready for use by all downstream cells.


In [3]:
# ============================================================
# Cell 3 — Feature Audit Definitions
# Each entry defines what to test, what endpoints to call,
# and what the Senior agent should verify in the response.
# ============================================================

@dataclass
class AuditFeature:
    id: str
    name: str
    tab: str
    endpoints: list          # List of (method, path, payload) tuples
    junior_mission: str      # Prompt for Junior: generate edge-case payloads
    senior_criteria: str     # Prompt for Senior: grade the API response
    expect_fields: list      # JSON fields that MUST appear in responses
    expect_status: list      # HTTP status codes that are acceptable

FEATURES = [

    AuditFeature(
        id="PROJ-01",
        name="Project Management",
        tab="Projects",
        endpoints=[
            ("POST",   "/api/v1/projects",        {"part_number": "TEST-PN-001", "revision": "A", "description": "Audit smoke test project"}),
            ("GET",    "/api/v1/projects",        None),
            ("PATCH",  "/api/v1/projects/{id}",   {"description": "Updated by audit"}),
            ("DELETE", "/api/v1/projects/{id}",   None),
        ],
        junior_mission=(
            "You are testing the MetroStack Project Management API endpoint at {base_url}.\n"
            "Generate 3 edge-case POST payloads for /api/v1/projects that test:\n"
            "1. A missing required field (expect 422)\n"
            "2. A duplicate part_number (expect 409 or 400)\n"
            "3. A valid project with maximum field lengths.\n"
            "Return ONLY a JSON array of objects with keys: payload, expect_status, rationale."
        ),
        senior_criteria=(
            "You are a senior QA engineer reviewing MetroStack API test results.\n"
            "Evaluate the following API responses for PROJ-01 Project Management:\n"
            "Required behaviours:\n"
            "  - POST creates a project and returns id, part_number, revision, description, status\n"
            "  - status field on creation must be 'pending'\n"
            "  - GET /projects returns a list with correct schema\n"
            "  - PATCH updates fields without destroying others\n"
            "  - DELETE returns 200 or 204 and removes the record\n"
            "  - Missing required fields must return 422, not 500\n"
            "Grade each sub-feature as PASS, PARTIAL, or FAIL with one-line reason.\n"
            "End with an overall score out of 5 in format: SCORE: X/5"
        ),
        expect_fields=["id", "part_number", "revision", "status"],
        expect_status=[200, 201, 204]
    ),

    AuditFeature(
        id="FILE-01",
        name="File / Scan Ingest",
        tab="Scans",
        endpoints=[
            ("GET",  "/api/v1/scans",           None),
            ("POST", "/api/v1/scans/ingest",    "__multipart__"),   # flagged for multipart handling
            ("GET",  "/api/v1/scans/{id}",      None),
        ],
        junior_mission=(
            "You are testing the MetroStack Scan Ingest API.\n"
            "The ingest endpoint accepts a multipart .e57 file upload at POST /api/v1/scans/ingest.\n"
            "Since we cannot upload a real file, generate 3 probe tests:\n"
            "1. POST with no file attached (expect 422)\n"
            "2. POST with wrong content type (expect 415 or 422)\n"
            "3. GET /api/v1/scans to verify list schema.\n"
            "Return JSON array with: endpoint, method, payload, expect_status, rationale."
        ),
        senior_criteria=(
            "Review MetroStack FILE-01 Scan Ingest test results.\n"
            "Required behaviours:\n"
            "  - GET /scans returns list with id, project_id, filename, point_count, status fields\n"
            "  - Ingest with no file returns 422, not 500\n"
            "  - Ingest with wrong type returns 415 or 422\n"
            "  - point_count field is numeric when scan is processed\n"
            "Grade each sub-feature PASS/PARTIAL/FAIL with reason.\n"
            "End with SCORE: X/5"
        ),
        expect_fields=["id", "filename", "point_count", "status"],
        expect_status=[200, 201, 204, 422]
    ),

    AuditFeature(
        id="ALIGN-01",
        name="Alignment",
        tab="Alignment",
        endpoints=[
            ("GET",  "/api/v1/alignments",           None),
            ("POST", "/api/v1/alignments",           {"scan_id": "__test_id__", "method": "best_fit", "z_offset": 0.0}),
            ("GET",  "/api/v1/alignments/{id}",      None),
        ],
        junior_mission=(
            "You are testing the MetroStack Alignment API.\n"
            "Alignment takes a scan_id and alignment method (best_fit, pin_bore, parting_plane).\n"
            "Generate 3 edge-case tests:\n"
            "1. POST with invalid scan_id (expect 404 or 422)\n"
            "2. POST with unsupported alignment method (expect 422)\n"
            "3. POST with z_offset as a string instead of float (expect 422).\n"
            "Return JSON array with: endpoint, method, payload, expect_status, rationale."
        ),
        senior_criteria=(
            "Review MetroStack ALIGN-01 Alignment test results.\n"
            "Required behaviours:\n"
            "  - POST /alignments accepts scan_id, method, optional z_offset\n"
            "  - Supported methods: best_fit, pin_bore, parting_plane\n"
            "  - Response includes alignment_matrix or transform fields\n"
            "  - Invalid scan_id returns 404, not 500\n"
            "  - Bad method returns 422 with clear message\n"
            "  - z_offset type validation works correctly\n"
            "Grade PASS/PARTIAL/FAIL with reason. End with SCORE: X/5"
        ),
        expect_fields=["id", "scan_id", "method", "status"],
        expect_status=[200, 201, 404, 422]
    ),

    AuditFeature(
        id="DEV-01",
        name="Deviation Analysis",
        tab="Deviations",
        endpoints=[
            ("GET",  "/api/v1/deviations",              None),
            ("POST", "/api/v1/deviations/compute",      {"scan_id": "__test_id__", "alignment_id": "__test_id__"}),
            ("GET",  "/api/v1/deviations/{id}/stats",   None),
        ],
        junior_mission=(
            "You are testing the MetroStack Deviation Analysis API.\n"
            "Deviation compute takes a scan_id and alignment_id and returns per-point statistics.\n"
            "Generate 3 edge-case tests:\n"
            "1. POST compute with mismatched scan/alignment IDs (expect 404 or 400)\n"
            "2. GET stats on a non-existent deviation ID (expect 404)\n"
            "3. GET /deviations with a filter for min_deviation > max_deviation (expect 422 or empty).\n"
            "Return JSON array with: endpoint, method, payload, expect_status, rationale."
        ),
        senior_criteria=(
            "Review MetroStack DEV-01 Deviation Analysis test results.\n"
            "Required behaviours:\n"
            "  - POST /deviations/compute triggers analysis and returns job id or result\n"
            "  - GET /deviations/{id}/stats returns: min, max, mean, stddev, point_count\n"
            "  - All stat fields are numeric (float)\n"
            "  - Invalid IDs return 404, not 500\n"
            "  - Filter validation works (422 on impossible ranges)\n"
            "Grade PASS/PARTIAL/FAIL. End with SCORE: X/5"
        ),
        expect_fields=["min", "max", "mean", "stddev", "point_count"],
        expect_status=[200, 201, 404, 422]
    ),

    AuditFeature(
        id="GDT-01",
        name="GD&T Feature Evaluation",
        tab="GD&T",
        endpoints=[
            ("GET",  "/api/v1/gdt/features",              None),
            ("POST", "/api/v1/gdt/features",              {"scan_id": "__test_id__", "feature_type": "flatness", "tolerance": 0.05, "datum": "A"}),
            ("GET",  "/api/v1/gdt/features/{id}/result", None),
        ],
        junior_mission=(
            "You are testing the MetroStack GD&T Feature Evaluation API.\n"
            "Supported feature types should include: flatness, straightness, circularity, cylindricity, parallelism, perpendicularity, position.\n"
            "Generate 3 edge-case tests:\n"
            "1. POST with an unsupported feature_type (expect 422)\n"
            "2. POST with negative tolerance value (expect 422)\n"
            "3. POST with tolerance=0.0 (edge: should this be valid or rejected?).\n"
            "Return JSON array with: endpoint, method, payload, expect_status, rationale."
        ),
        senior_criteria=(
            "Review MetroStack GDT-01 GD&T Feature Evaluation test results.\n"
            "Required behaviours:\n"
            "  - POST /gdt/features accepts feature_type, tolerance, datum reference\n"
            "  - Supported types include at minimum: flatness, circularity, position\n"
            "  - Result endpoint returns: actual_value, tolerance, pass_fail, deviation_map reference\n"
            "  - Negative tolerance returns 422\n"
            "  - Unsupported type returns 422 with descriptive error\n"
            "  - Datum chaining doesn't crash (A->B->C references)\n"
            "Grade PASS/PARTIAL/FAIL. End with SCORE: X/5"
        ),
        expect_fields=["id", "feature_type", "tolerance", "pass_fail"],
        expect_status=[200, 201, 422]
    ),

    AuditFeature(
        id="WALL-01",
        name="Wall Thickness Analysis",
        tab="Wall Analysis",
        endpoints=[
            ("GET",  "/api/v1/wall",              None),
            ("POST", "/api/v1/wall/analyze",      {"scan_id": "__test_id__", "nominal_thickness": 4.0, "tolerance_plus": 0.5, "tolerance_minus": 0.5}),
            ("GET",  "/api/v1/wall/{id}/bands",   None),
        ],
        junior_mission=(
            "You are testing the MetroStack Wall Thickness Analysis API (Giga Press mold context).\n"
            "Wall analysis computes thickness deviation across a parting surface scan.\n"
            "Generate 3 edge-case tests:\n"
            "1. POST analyze with tolerance_plus < 0 (expect 422)\n"
            "2. POST analyze with nominal_thickness = 0 (edge case — valid or rejected?)\n"
            "3. GET /wall/{id}/bands on unprocessed scan (expect 404 or 425 Too Early).\n"
            "Return JSON array with: endpoint, method, payload, expect_status, rationale."
        ),
        senior_criteria=(
            "Review MetroStack WALL-01 Wall Thickness Analysis test results.\n"
            "Required behaviours (Giga Press mold parting surface context):\n"
            "  - POST /wall/analyze accepts scan_id, nominal_thickness, tolerance_plus, tolerance_minus\n"
            "  - Response includes: analysis_id, status, section_count\n"
            "  - GET /wall/{id}/bands returns per-section pass/fail bands\n"
            "  - Each band has: section_id, measured, nominal, deviation, pass_fail\n"
            "  - Negative tolerances return 422\n"
            "  - Unprocessed scan returns 404 or 425, not 500\n"
            "Grade PASS/PARTIAL/FAIL. End with SCORE: X/5"
        ),
        expect_fields=["analysis_id", "status", "section_count"],
        expect_status=[200, 201, 404, 422]
    ),

    AuditFeature(
        id="VIZ-01",
        name="Visualization / Colour Map",
        tab="Visualization",
        endpoints=[
            ("GET",  "/api/v1/viz/colormap",         None),
            ("POST", "/api/v1/viz/colormap",         {"deviation_id": "__test_id__", "range_min": -1.0, "range_max": 1.0, "palette": "blue_red"}),
            ("GET",  "/api/v1/viz/export/{id}",      None),
        ],
        junior_mission=(
            "You are testing the MetroStack Visualization / Colour Map API.\n"
            "The colormap endpoint generates a visual deviation overlay from a deviation analysis result.\n"
            "Generate 3 edge-case tests:\n"
            "1. POST colormap with range_min > range_max (expect 422)\n"
            "2. POST colormap with unsupported palette name (expect 422)\n"
            "3. GET /viz/export/{id} with non-existent id (expect 404).\n"
            "Return JSON array with: endpoint, method, payload, expect_status, rationale."
        ),
        senior_criteria=(
            "Review MetroStack VIZ-01 Visualization test results.\n"
            "Required behaviours:\n"
            "  - POST /viz/colormap accepts deviation_id, range_min, range_max, palette\n"
            "  - Supported palettes include at minimum: blue_red, green_red, greyscale\n"
            "  - Response includes: viz_id, point_count, range_min, range_max, tile_url or data_url\n"
            "  - range_min > range_max returns 422\n"
            "  - Invalid palette returns 422 with list of valid options\n"
            "  - Export endpoint returns file or signed URL\n"
            "Grade PASS/PARTIAL/FAIL. End with SCORE: X/5"
        ),
        expect_fields=["viz_id", "range_min", "range_max"],
        expect_status=[200, 201, 404, 422]
    ),

]

print(f"\u2713 {len(FEATURES)} feature audit definitions loaded.")
for f in FEATURES:
    print(f"  [{f.id}] {f.name} — Tab: {f.tab} — {len(f.endpoints)} endpoint(s)")

✓ 7 feature audit definitions loaded.
  [PROJ-01] Project Management — Tab: Projects — 4 endpoint(s)
  [FILE-01] File / Scan Ingest — Tab: Scans — 3 endpoint(s)
  [ALIGN-01] Alignment — Tab: Alignment — 3 endpoint(s)
  [DEV-01] Deviation Analysis — Tab: Deviations — 3 endpoint(s)
  [GDT-01] GD&T Feature Evaluation — Tab: GD&T — 3 endpoint(s)
  [WALL-01] Wall Thickness Analysis — Tab: Wall Analysis — 3 endpoint(s)
  [VIZ-01] Visualization / Colour Map — Tab: Visualization — 3 endpoint(s)


In [4]:
# ============================================================
# Cell 4 — HTTP Client (Jupyter + Pyodide compatible)
# ============================================================

import json

def http_call(method, url, payload=None, timeout=10):
    import urllib.request, urllib.error
    try:
        data = json.dumps(payload).encode() if payload and payload != "__multipart__" else None
        req  = urllib.request.Request(
            url, data=data, method=method,
            headers={"Content-Type": "application/json"}
        )
        with urllib.request.urlopen(req, timeout=timeout) as r:
            body = json.loads(r.read().decode())
        return {"status": r.status, "body": body, "error": None}
    except urllib.error.HTTPError as e:
        try:    body = json.loads(e.read().decode())
        except: body = {"raw": str(e)}
        return {"status": e.code, "body": body, "error": None}
    except urllib.error.URLError as e:
        return {"status": 0, "body": {}, "error": str(e.reason)}
    except Exception as e:
        return {"status": 0, "body": {}, "error": str(e)}


def query_llm(base_url, role_name, system_prompt, user_prompt, max_tokens=900):
    import urllib.request, urllib.error
    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        "temperature": 0.2, "top_p": 0.85, "max_tokens": max_tokens
    }
    try:
        data = json.dumps(payload).encode()
        req  = urllib.request.Request(
            f"{base_url}/v1/chat/completions", data=data,
            headers={"Content-Type": "application/json"}
        )
        with urllib.request.urlopen(req, timeout=60) as r:
            return json.loads(r.read().decode())["choices"][0]["message"]["content"]
    except urllib.error.URLError as e:
        return f"__CONN_ERR__ {role_name} at {base_url}: {e.reason}"
    except Exception as e:
        return f"__ERR__ {str(e)[:200]}"


def parse_score(senior_output: str) -> float:
    """Extract numeric score from Senior agent output. Returns 0.0-1.0."""
    match = re.search(r'SCORE:\s*(\d+)\s*/\s*(\d+)', senior_output, re.IGNORECASE)
    if match:
        num, denom = int(match.group(1)), int(match.group(2))
        return num / denom if denom > 0 else 0.0
    return 0.0


def letter_grade(score: float) -> str:
    if score >= 0.9: return "A"
    if score >= 0.8: return "B"
    if score >= 0.65: return "C"
    if score >= 0.4: return "D"
    return "F"


print("\u2713 HTTP client and LLM query functions loaded.")

✓ HTTP client and LLM query functions loaded.


In [5]:
# ============================================================
# Cell 5 — Pre-Flight Health Check
# Verify MetroStack API and both LLM agents are reachable
# before running the full audit suite.
# ============================================================

print("Running pre-flight health checks...\n")

checks = [
    ("MetroStack API",  METROSTACK_URL, "/docs"),
    ("Junior Agent",    JUNIOR_URL,     "/v1/models"),
    ("Senior Agent",    SENIOR_URL,     "/v1/models"),
]

all_ok = True
for label, base, path in checks:
    result = http_call("GET", f"{base}{path}", timeout=5)
    if result["error"]:
        status_str = f"\u2717 OFFLINE — {result['error']}"
        all_ok = False
    elif result["status"] in [200, 201]:
        status_str = f"\u2713 ONLINE (HTTP {result['status']})"
    else:
        status_str = f"\u26a0 HTTP {result['status']} — may still function"
    print(f"  {label:20s}  {base:35s}  {status_str}")

print()
if all_ok:
    print("\u2713 All systems ready. Proceed to Cell 5 to run the audit.")
else:
    print("\u26a0 One or more services offline.")
    print("  - For LLM agents: ensure llama-server.exe or Ollama is running with --cors-allow-origin '*'")
    print("  - For MetroStack: ensure uvicorn is running at the configured port.")
    print("  The audit will still run but offline services will log __CONN_ERR__ results.")
# ── Dataset context status ────────────────────────────────────────────────────
print()
print("  Dataset Context (from scanner cell):")
print(f"    project_ids   : {len(DATASET_CONTEXT['project_ids'])} loaded")
print(f"    scan_ids      : {len(DATASET_CONTEXT['scan_ids'])} loaded")
print(f"    alignment_ids : {len(DATASET_CONTEXT['alignment_ids'])} loaded")
print(f"    deviation_ids : {len(DATASET_CONTEXT['deviation_ids'])} loaded")
print(f"    point_clouds  : {len(DATASET_CONTEXT['point_cloud_files'])} file(s) registered")
no_ids = not any([
    DATASET_CONTEXT['project_ids'], DATASET_CONTEXT['scan_ids'],
    DATASET_CONTEXT['alignment_ids'], DATASET_CONTEXT['deviation_ids']
])
if no_ids:
    print()
    print("  ⚠ No real IDs loaded — tests will use synthetic placeholders.")
    print("    Upload MetroStack fixture exports via ENVIRONMENT FILES")
    print("    for live record-level validation.")


Running pre-flight health checks...

Traceback (most recent call last):
  File "/lib/python312.zip/_pyodide/_base.py", line 597, in eval_code_async
    await CodeRunner(
  File "/lib/python312.zip/_pyodide/_base.py", line 411, in run_async
    coroutine = eval(self.code, globals, locals)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<exec>", line 17, in <module>
  File "<exec>", line 13, in http_call
ModuleNotFoundError: No module named 'requests'
The module 'requests' is included in the Pyodide distribution, but it is not installed.
You can install it by calling:
  await micropip.install("requests") in Python, or
  await pyodide.loadPackage("requests") in JavaScript
See https://pyodide.org/en/stable/usage/loading-packages.html for more details.


NoneType: None

In [6]:
# NOTE: get_test_id() and DATASET_CONTEXT come from the dataset scanner cell.

# ============================================================
# Cell 5 — Core Audit Engine
# For each feature:
#   1. Run baseline endpoint calls against MetroStack
#   2. Ask Junior to generate edge-case payloads
#   3. Run Junior's edge cases against MetroStack
#   4. Send all results to Senior for grading
#   5. Parse score and store result
# ============================================================

@dataclass
class AuditResult:
    feature_id: str
    feature_name: str
    baseline_results: list = field(default_factory=list)
    junior_edge_cases: list = field(default_factory=list)
    edge_case_results: list = field(default_factory=list)
    senior_verdict: str = ""
    score: float = 0.0
    grade: str = "?"
    status: str = "PENDING"
    latency_s: float = 0.0


def run_baseline_calls(feature: AuditFeature, created_id: Optional[str] = None) -> list:
    """Execute the baseline endpoint calls defined in the feature spec."""
    results = []
    for method, path, payload in feature.endpoints:
        if "__multipart__" in str(payload):
            results.append({"method": method, "path": path, "status": "SKIPPED", "note": "multipart — file upload skipped in smoke test"})
            continue

        # Substitute placeholder IDs with created ID if available
        actual_path = path.replace("{id}", created_id or "nonexistent-id-000")
        if isinstance(payload, dict):
            payload_str = json.dumps(payload).replace('"__test_id__"', f'"{created_id or "nonexistent-id-000"}"}"')
            try:
                actual_payload = json.loads(payload_str)
            except Exception:
                actual_payload = payload
        else:
            actual_payload = payload

        result = http_call(method, f"{METROSTACK_URL}{actual_path}", actual_payload)

        # Capture created ID from POST response for use in PATCH/DELETE/GET-by-id calls
        if method == "POST" and result["status"] in [200, 201]:
            body = result.get("body", {})
            if isinstance(body, dict):
                created_id = body.get("id") or body.get("project_id") or created_id

        results.append({
            "method": method,
            "path": actual_path,
            "status": result["status"],
            "body": result["body"],
            "error": result["error"]
        })
    return results


def run_edge_cases(cases: list) -> list:
    """Execute edge-case calls generated by Junior agent."""
    results = []
    for case in cases:
        if not isinstance(case, dict):
            continue
        method  = case.get("method", "GET")
        path    = case.get("endpoint", case.get("path", "/"))
        payload = case.get("payload")
        expect  = case.get("expect_status", 200)

        result = http_call(method, f"{METROSTACK_URL}{path}", payload)
        met = result["status"] == expect or (result["status"] == 0 and result["error"])

        results.append({
            "method": method,
            "path": path,
            "got_status": result["status"],
            "expect_status": expect,
            "expectation_met": met,
            "rationale": case.get("rationale", ""),
            "error": result["error"]
        })
    return results


JUNIOR_SYSTEM = (
    "You are MetroBot Junior, a code and API testing assistant for the MetroStack metrology platform.\n"
    "MetroStack is a FastAPI + PostgreSQL + PostGIS platform for 3D scan deviation analysis.\n"
    "When asked to generate test cases, return ONLY valid JSON — no markdown fences, no explanation."
)

SENIOR_SYSTEM = (
    "You are MetroBot Senior, a senior QA engineer and architect for the MetroStack metrology platform.\n"
    "You review raw API test results and render structured verdicts.\n"
    "Always end your response with a line in this exact format: SCORE: X/5"
)


def audit_feature(feature: AuditFeature) -> AuditResult:
    result = AuditResult(feature_id=feature.id, feature_name=feature.name)
    t_start = time.time()

    print(f"\n{'='*70}")
    print(f"  Auditing: {feature.id} — {feature.name}")
    print(f"{'='*70}")

    # Step 1: Baseline calls
    print("  [1/4] Running baseline endpoint calls...")
    result.baseline_results = run_baseline_calls(feature)
    for r in result.baseline_results:
        flag = "\u2713" if r.get("status") in feature.expect_status or r.get("status") == "SKIPPED" else "\u26a0"
        print(f"        {flag} {r.get('method','?'):6s} {r.get('path','')} -> {r.get('status','')} {r.get('error') or ''}")

    # Step 2: Junior generates edge cases
    print("  [2/4] Asking Junior to generate edge-case payloads...")
    junior_prompt = feature.junior_mission.format(base_url=METROSTACK_URL)
    junior_raw = query_llm(JUNIOR_URL, "Junior", JUNIOR_SYSTEM, junior_prompt, max_tokens=600)

    if "__CONN_ERR__" in junior_raw or "__ERR__" in junior_raw:
        print(f"        \u2717 Junior agent error: {junior_raw[:80]}")
        result.junior_edge_cases = []
    else:
        clean = junior_raw.strip().lstrip('`').rstrip('`')
        if clean.startswith('json'): clean = clean[4:].strip()
        try:
            cases = json.loads(clean)
            result.junior_edge_cases = cases if isinstance(cases, list) else []
            print(f"        \u2713 Junior generated {len(result.junior_edge_cases)} edge case(s).")
        except json.JSONDecodeError:
            print(f"        \u26a0 Junior response not valid JSON — skipping edge cases.")
            result.junior_edge_cases = []

    # Step 3: Run edge cases
    if result.junior_edge_cases:
        print("  [3/4] Running Junior's edge-case calls against MetroStack...")
        result.edge_case_results = run_edge_cases(result.junior_edge_cases)
        for ec in result.edge_case_results:
            met_str = "\u2713 MET" if ec["expectation_met"] else "\u2717 MISS"
            print(f"        {met_str} | {ec['method']:6s} {ec['path']} | got {ec['got_status']} expected {ec['expect_status']}")
    else:
        print("  [3/4] No edge cases to run.")
        result.edge_case_results = []

    # Step 4: Senior grades everything
    print("  [4/4] Sending results to Senior agent for grading...")
    combined_log = json.dumps({
        "feature": feature.id,
        "baseline": result.baseline_results,
        "edge_cases": result.edge_case_results
    }, indent=2)
    senior_user_prompt = f"{feature.senior_criteria}\n\n--- RAW TEST RESULTS ---\n{combined_log}"
    result.senior_verdict = query_llm(SENIOR_URL, "Senior", SENIOR_SYSTEM, senior_user_prompt, max_tokens=900)

    if "__CONN_ERR__" in result.senior_verdict or "__ERR__" in result.senior_verdict:
        print(f"        \u2717 Senior agent error: {result.senior_verdict[:80]}")
        result.score = 0.0
        result.grade = "?"
        result.status = "AGENT_ERROR"
    else:
        result.score = parse_score(result.senior_verdict)
        result.grade = letter_grade(result.score)
        result.status = "PASS" if result.score >= PASS_THRESHOLD else ("PARTIAL" if result.score >= RETRY_THRESHOLD else "FAIL")
        print(f"        Senior verdict: {result.score:.0%} | Grade: {result.grade} | {result.status}")

    result.latency_s = time.time() - t_start
    return result


print("\u2713 Audit engine ready. Run Cell 6 to execute.")

Traceback (most recent call last):
  File "/lib/python312.zip/_pyodide/_base.py", line 597, in eval_code_async
    await CodeRunner(
          ^^^^^^^^^^^
  File "/lib/python312.zip/_pyodide/_base.py", line 285, in __init__
    self.ast = next(self._gen)
               ^^^^^^^^^^^^^^^
  File "/lib/python312.zip/_pyodide/_base.py", line 149, in _parse_and_compile_gen
    mod = compile(source, filename, mode, flags | ast.PyCF_ONLY_AST)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<exec>", line 38
    payload_str = json.dumps(payload).replace('"__test_id__"', f'"{created_id or "nonexistent-id-000"}"}"')
                                                                                                       ^
SyntaxError: f-string: single '}' is not allowed


NoneType: None

In [7]:
# ============================================================
# Cell 6 — Run Audit Pipeline
# Executes all 7 feature audits in sequence.
# Results are stored in AUDIT_RESULTS for the report cell.
# ============================================================

AUDIT_RESULTS = []

print("=" * 70)
print("  METROSTACK LIVE FUNCTIONAL AUDIT")
print(f"  Target   : {METROSTACK_URL}")
print(f"  Junior   : {JUNIOR_URL}")
print(f"  Senior   : {SENIOR_URL}")
print(f"  Features : {len(FEATURES)}")
print("=" * 70)

t_total = time.time()

for feature in FEATURES:
    result = audit_feature(feature)
    AUDIT_RESULTS.append(result)

print(f"\n\n{'='*70}")
print(f"  Audit complete. Total time: {time.time() - t_total:.1f}s")
print(f"  Results summary in Cell 7.")
print(f"{'='*70}")

  METROSTACK LIVE FUNCTIONAL AUDIT
  Target   : http://localhost:8000
  Junior   : http://localhost:8080
  Senior   : http://localhost:8081
  Features : 7
Traceback (most recent call last):
  File "/lib/python312.zip/_pyodide/_base.py", line 597, in eval_code_async
    await CodeRunner(
  File "/lib/python312.zip/_pyodide/_base.py", line 411, in run_async
    coroutine = eval(self.code, globals, locals)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<exec>", line 20, in <module>
NameError: name 'audit_feature' is not defined. Did you mean: 'AuditFeature'?


NoneType: None

In [8]:
# ============================================================
# Cell 7 — Final Report & Performance Grid
# ============================================================

pass_count    = sum(1 for r in AUDIT_RESULTS if r.status == "PASS")
partial_count = sum(1 for r in AUDIT_RESULTS if r.status == "PARTIAL")
fail_count    = sum(1 for r in AUDIT_RESULTS if r.status in ["FAIL", "AGENT_ERROR"])
avg_score     = sum(r.score for r in AUDIT_RESULTS) / len(AUDIT_RESULTS) if AUDIT_RESULTS else 0

print()
print("=" * 70)
print("  METROSTACK AUDIT REPORT")
print("=" * 70)
print(f"{'Feature ID':<12} {'Name':<30} {'Score':>6} {'Grade':>6} {'Status':<12} {'Time':>7}")
print("-" * 70)

for r in AUDIT_RESULTS:
    status_icon = {"PASS": "\u2713 PASS", "PARTIAL": "~ PARTIAL", "FAIL": "\u2717 FAIL", "AGENT_ERROR": "? ERROR"}.get(r.status, r.status)
    print(f"{r.feature_id:<12} {r.feature_name:<30} {r.score:>5.0%} {r.grade:>6} {status_icon:<12} {r.latency_s:>5.1f}s")

print("-" * 70)
print(f"  Overall: {avg_score:.0%} | PASS: {pass_count}  PARTIAL: {partial_count}  FAIL: {fail_count}")
print("=" * 70)

print("\n\n--- SENIOR VERDICTS ---\n")
for r in AUDIT_RESULTS:
    print(f"\n[{r.feature_id}] {r.feature_name}")
    print("-" * 50)
    if r.senior_verdict and "__" not in r.senior_verdict[:5]:
        print(r.senior_verdict)
    else:
        print(f"  Agent error or no response: {r.senior_verdict[:100]}")
    print()


  METROSTACK AUDIT REPORT
Feature ID   Name                            Score  Grade Status          Time
----------------------------------------------------------------------
----------------------------------------------------------------------
  Overall: 0% | PASS: 0  PARTIAL: 0  FAIL: 0


--- SENIOR VERDICTS ---

